# SUPPORT on 10 data splits

In notebooks 10, 14 and 15 we only used 3 data splits, which is too few to tell if the differences between methods are real or just luck. Here we run the main SUPPORT experiments again on 10 splits (split 42 as before, plus splits 1-9).

What we check:
1. how well the classical models (Cox, RSF, Weibull AFT, DSM) do,
2. how much the models lose without the post-baseline variables,
3. how the way of handling missing values changes VaDeSC (VAE imputation, mean imputation, missingness-aware VaDeSC),
4. if VaDeSC gets better when we tune its settings.

We use SUPPORT without the post-baseline variables, unless we say otherwise.

All runs on the same split are tested on the same patients, so for the tests we first average the seeds within each split and get one value per split. We make many comparisons, so we use the Bonferroni correction (each p-value is multiplied by the number of comparisons).

Running everything takes about 1.5 hours on a CPU. Set `QUICK = True` for a short test.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import torch
from joblib import Parallel, delayed
from scipy import stats
from sksurv.metrics import concordance_index_censored

from imputation.vae_impute import load_support_from_frame, vae_impute_support
from src.baselines import run_cox, run_dsm, run_rsf, run_weibull_aft
from src.data import load_support
from src.missing import MissVaDeSC, fill_and_mask
from src.vadesc import VaDeSC, evaluate_vadesc, tensor, train_vadesc

QUICK = False
SPLITS = [42, 1] if QUICK else [42, 1, 2, 3, 4, 5, 6, 7, 8, 9]
SEEDS = [0, 1] if QUICK else [0, 1, 2, 3, 4]
LEAK_SEEDS = [0] if QUICK else [0, 1, 2]
TUNE_SEEDS = [0] if QUICK else [0, 1]
EPOCHS = 30 if QUICK else 300
N_JOBS = 8
PAPER = dict(K=4, d_lat=16, weibull_k=2.0)


class VaDeSCLearnK(VaDeSC):
    # same model, but the Weibull shape k is learned (initialised at weibull_k), as in notebook 14
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.learn_k = True


def by_split(df, value="C-index"):
    # one value per split and method (mean over seeds)
    return df.groupby(["method", "split"])[value].mean().unstack("method")


def split_test(per_split, a, b, n_comp=1):
    # paired t-test over splits; Bonferroni-corrected p-value for n_comp comparisons
    diff = per_split[a] - per_split[b]
    n = len(diff)
    ci = stats.t.ppf(0.975, n - 1) * diff.std(ddof=1) / np.sqrt(n)
    p = stats.ttest_1samp(diff, 0).pvalue
    return {"comparison": f"{a}  minus  {b}", "mean diff": diff.mean(),
            "95% CI": f"[{diff.mean() - ci:+.3f}, {diff.mean() + ci:+.3f}]",
            "higher in": f"{(diff > 0).sum()}/{n} splits", "p-value": p, "p (Bonferroni)": min(1.0, p * n_comp)}

## 1. Data

For each split we make all the versions of the data that we need: four simple imputations (median or mean for numbers, most frequent value or a separate "missing" category for categories), VAE imputation (the VAE is trained again on each split, as in notebook 10), the data with the missingness mask, and the data with the post-baseline variables.

In [2]:
FILL = {"median/mode": ("median", "most_frequent"), "mean/missing category": ("mean", "missing"),
        "median/missing category": ("median", "missing"), "mean/mode": ("mean", "most_frequent")}


def make_vae(sp):
    torch.set_num_threads(2)
    return sp, load_support_from_frame(vae_impute_support(drop_post_baseline=True, split_seed=sp), drop_post_baseline=True, seed=sp)


data = {}
for sp in SPLITS:
    for name, (nf, cf) in FILL.items():
        data[(sp, name)] = load_support(drop_post_baseline=True, seed=sp, num_fill=nf, cat_fill=cf)
    data[(sp, "mask")] = fill_and_mask(load_support(drop_post_baseline=True, seed=sp, keep_missing=True), True)
    data[(sp, "with post-baseline")] = load_support(drop_post_baseline=False, seed=sp)
for sp, d in Parallel(n_jobs=N_JOBS)(delayed(make_vae)(sp) for sp in SPLITS):
    data[(sp, "VAE")] = d
print({k[1]: v.X_tr.shape for k, v in data.items() if k[0] == 42})

{'median/mode': (6373, 51), 'mean/missing category': (6373, 53), 'median/missing category': (6373, 53), 'mean/mode': (6373, 51), 'mask': (6373, 106), 'with post-baseline': (6373, 59), 'VAE': (6373, 51)}


## 2. Classical models

Cox, RSF, Weibull AFT and DSM on every split, with median/mode imputation. DSM chooses its own settings on the validation set, as in notebook 09.

In [3]:
def run_classical(sp, d):
    torch.set_num_threads(2)
    out = []
    for model, fn in [("Cox PH", run_cox), ("RSF", lambda x: run_rsf(x, seed=sp)), ("Weibull AFT", run_weibull_aft),
                      ("DSM", lambda x: run_dsm(x, seed=sp))]:
        m = fn(d)
        out.append({"split": sp, "method": model, "C-index": m["C-index"], "IBS": m["IBS"]})
    return out


cls = pd.DataFrame([r for rows in Parallel(n_jobs=2)(delayed(run_classical)(sp, data[(sp, "median/mode")]) for sp in SPLITS)
                    for r in rows])
cls.to_csv("../results/support10_classical.csv", index=False)
cls.groupby("method")[["C-index", "IBS"]].agg(["mean", "std"]).round(3)

C-index           IBS       
               mean    std   mean    std
method                                  
Cox PH        0.731  0.009  0.177  0.004
DSM           0.734  0.009  0.174  0.004
RSF           0.727  0.010  0.174  0.003
Weibull AFT   0.728  0.009  0.178  0.003

## 3. With and without the post-baseline variables

Cox and RSF once per split, VaDeSC with the paper settings and 3 seeds per split.

In [4]:
def run_vadesc(d, seed, label, sp, model_cls=VaDeSC, model_kwargs=None, **cfg):
    torch.set_num_threads(2)
    cfg = {**PAPER, **cfg}
    model = train_vadesc(d, epochs=EPOCHS, seed=seed, model_cls=model_cls,
                         model_kwargs={"net": "small", **(model_kwargs or {})}, **cfg)
    m, _, _ = evaluate_vadesc(model, d)
    risk_va = model.risk_score(tensor(d.X_va)).cpu().numpy()
    val_c = concordance_index_censored(d.e_va.astype(bool), d.t_va, risk_va)[0]
    k_hat = float(model.shape()) if cfg.get("dist", "weibull") == "weibull" else np.nan
    return {"split": sp, "seed": seed, "method": label, "C-index": m["C-index"], "IBS": m["IBS"],
            "log-rank chi2": m["log-rank chi2"], "val C-index": val_c, "k": k_hat}


leak_rows = []
for sp in SPLITS:
    d = data[(sp, "with post-baseline")]
    leak_rows += [{"split": sp, "method": "Cox PH, with", "C-index": run_cox(d)["C-index"]},
                  {"split": sp, "method": "RSF, with", "C-index": run_rsf(d, seed=sp)["C-index"]}]
leak_rows += Parallel(n_jobs=N_JOBS)(delayed(run_vadesc)(data[(sp, "with post-baseline")], s, "VaDeSC, with", sp)
                                     for sp in SPLITS for s in LEAK_SEEDS)
leak_rows += Parallel(n_jobs=N_JOBS)(delayed(run_vadesc)(data[(sp, "median/mode")], s, "VaDeSC, without", sp)
                                     for sp in SPLITS for s in LEAK_SEEDS)
leak = pd.concat([pd.DataFrame(leak_rows),
                  cls[cls.method.isin(["Cox PH", "RSF"])].assign(method=lambda x: x.method + ", without")])
leak.to_csv("../results/support10_leakage.csv", index=False)
leak_split = by_split(leak)
pd.DataFrame({"mean": leak_split.mean(), "std over splits": leak_split.std()}).round(3)

,mean,std over splits
method,,
"Cox PH, with",0.804,0.006
"Cox PH, without",0.731,0.009
"RSF, with",0.810,0.006
"RSF, without",0.727,0.010
"VaDeSC, with",0.775,0.009
"VaDeSC, without",0.663,0.011


## 4. Different ways of handling missing values (10 splits, 5 seeds)

All VaDeSC versions use the same network and the paper settings, so only the handling of missing values changes:
- median / most frequent (what we use everywhere else),
- median / own category, mean / most frequent, mean / own category (the "mean imputation" of notebook 15),
- VAE imputation,
- mean imputation + the mask as extra input,
- MissVaDeSC (mask as input, reconstructs only the observed values),
- MissVaDeSC + mask model (also has to predict the mask).

We also run Cox on every version of the data. Note: Cox with the mask has a much worse IBS on splits 3, 6 and 9 (about 0.32 instead of 0.18), while its C-index is normal there. We think the Cox fit becomes unstable with the extra mask columns.

In [5]:
VARIANTS = {
    "VaDeSC, median/mode": ("median/mode", {}),
    "VaDeSC, mean/missing category": ("mean/missing category", {}),
    "VaDeSC, median/missing category": ("median/missing category", {}),
    "VaDeSC, mean/mode": ("mean/mode", {}),
    "VaDeSC, VAE imputation": ("VAE", {}),
    "VaDeSC + mask input": ("mask", {}),
    "MissVaDeSC": ("mask", {"model_cls": MissVaDeSC, "model_kwargs": {"model_mask": False}}),
    "MissVaDeSC + mask model": ("mask", {"model_cls": MissVaDeSC, "model_kwargs": {"model_mask": True}}),
}
jobs = [(data[(sp, key)], s, name, sp, kw) for sp in SPLITS for s in SEEDS for name, (key, kw) in VARIANTS.items()]
miss = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=2)(delayed(run_vadesc)(d, s, name, sp, **kw) for d, s, name, sp, kw in jobs))
cox_imp = pd.DataFrame([{"split": sp, "method": f"Cox PH, {key}", "C-index": run_cox(data[(sp, key)])["C-index"],
                         "IBS": run_cox(data[(sp, key)])["IBS"]}
                        for sp in SPLITS for key in list(FILL) + ["VAE", "mask"]])
pd.concat([miss, cox_imp]).to_csv("../results/support10_missingness.csv", index=False)
summary = pd.concat([miss, cox_imp]).groupby("method", sort=False)[["C-index", "IBS"]].agg(["mean", "std"])
summary.round(3)

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed:  2.8min


[Parallel(n_jobs=8)]: Done 146 tasks      | elapsed: 14.0min


[Parallel(n_jobs=8)]: Done 349 tasks      | elapsed: 32.8min


[Parallel(n_jobs=8)]: Done 400 out of 400 | elapsed: 37.6min finished


C-index           IBS       
                                   mean    std   mean    std
method                                                      
VaDeSC, median/mode               0.658  0.016  0.207  0.007
VaDeSC, mean/missing category     0.684  0.016  0.195  0.005
VaDeSC, median/missing category   0.662  0.017  0.204  0.009
VaDeSC, mean/mode                 0.684  0.015  0.195  0.004
VaDeSC, VAE imputation            0.673  0.014  0.200  0.005
VaDeSC + mask input               0.665  0.014  0.199  0.004
MissVaDeSC                        0.671  0.014  0.197  0.004
MissVaDeSC + mask model           0.661  0.010  0.200  0.005
Cox PH, median/mode               0.731  0.009  0.177  0.004
Cox PH, mean/missing category     0.730  0.009  0.176  0.004
Cox PH, median/missing category   0.730  0.009  0.177  0.004
Cox PH, mean/mode                 0.730  0.009  0.176  0.004
Cox PH, VAE                       0.732  0.009  0.176  0.004
Cox PH, mask                      0.732  0.009  0.221  0.070

In [6]:
# tests over splits (one value per split = mean over the 5 seeds), Bonferroni within this section
ms = by_split(miss)
ref, mean_imp = "VaDeSC, median/mode", "VaDeSC, mean/missing category"
comps = [(m, ref) for m in VARIANTS if m != ref] + \
        [(m, mean_imp) for m in ["VaDeSC + mask input", "MissVaDeSC", "MissVaDeSC + mask model"]] + \
        [(mean_imp, "VaDeSC, VAE imputation"), ("MissVaDeSC", "VaDeSC, VAE imputation")]
miss_tests = pd.DataFrame([split_test(ms, a, b, len(comps)) for a, b in comps])
pd.set_option("display.max_colwidth", 90)
miss_tests.round(4)

,comparison,mean diff,95% CI,higher in,p-value,p (Bonferroni)
0,"VaDeSC, mean/missing category minus VaDeSC, median/mode",0.0256,"[+0.020, +0.031]",10/10 splits,0.0000,0.0000
1,"VaDeSC, median/missing category minus VaDeSC, median/mode",0.0043,"[-0.003, +0.011]",8/10 splits,0.2094,1.0000
2,"VaDeSC, mean/mode minus VaDeSC, median/mode",0.0259,"[+0.022, +0.030]",10/10 splits,0.0000,0.0000
3,"VaDeSC, VAE imputation minus VaDeSC, median/mode",0.0145,"[+0.009, +0.020]",10/10 splits,0.0002,0.0023
4,"VaDeSC + mask input minus VaDeSC, median/mode",0.0071,"[+0.001, +0.013]",9/10 splits,0.0208,0.2496
5,"MissVaDeSC minus VaDeSC, median/mode",0.0131,"[+0.008, +0.018]",10/10 splits,0.0002,0.0021
6,"MissVaDeSC + mask model minus VaDeSC, median/mode",0.0033,"[-0.002, +0.008]",8/10 splits,0.1720,1.0000
7,"VaDeSC + mask input minus VaDeSC, mean/missing category",-0.0185,"[-0.023, -0.014]",0/10 splits,0.0000,0.0001
8,"MissVaDeSC minus VaDeSC, mean/missing category",-0.0125,"[-0.018, -0.007]",1/10 splits,0.0009,0.0108
9,"MissVaDeSC + mask model minus VaDeSC, mean/missing category",-0.0223,"[-0.027, -0.017]",0/10 splits,0.0000,0.0000


### Why is mean imputation better?

Mean imputation is different from median/mode imputation in two ways: missing numbers get the mean instead of the median, and missing categories get their own category instead of the most frequent one. Here we change only one of the two at a time to see which one matters.

In [7]:
abl = [("VaDeSC, mean/mode", "VaDeSC, median/mode"), ("VaDeSC, mean/missing category", "VaDeSC, median/missing category"),
       ("VaDeSC, median/missing category", "VaDeSC, median/mode"), ("VaDeSC, mean/missing category", "VaDeSC, mean/mode")]
print("C-index, mean over splits (rows: numeric fill, columns: categorical handling)")
print(pd.DataFrame({"most frequent": [ms["VaDeSC, median/mode"].mean(), ms["VaDeSC, mean/mode"].mean()],
                    "own 'missing' category": [ms["VaDeSC, median/missing category"].mean(), ms["VaDeSC, mean/missing category"].mean()]},
                   index=["median", "mean"]).round(3))
pd.DataFrame([split_test(ms, a, b, len(abl)) for a, b in abl]).round(4)

C-index, mean over splits (rows: numeric fill, columns: categorical handling)
        most frequent  own 'missing' category
median          0.658                   0.662
mean            0.684                   0.684


,comparison,mean diff,95% CI,higher in,p-value,p (Bonferroni)
0,"VaDeSC, mean/mode minus VaDeSC, median/mode",0.0259,"[+0.022, +0.030]",10/10 splits,0.0000,0.0000
1,"VaDeSC, mean/missing category minus VaDeSC, median/missing category",0.0213,"[+0.014, +0.029]",10/10 splits,0.0001,0.0006
2,"VaDeSC, median/missing category minus VaDeSC, median/mode",0.0043,"[-0.003, +0.011]",8/10 splits,0.2094,0.8376
3,"VaDeSC, mean/missing category minus VaDeSC, mean/mode",-0.0003,"[-0.005, +0.004]",6/10 splits,0.8843,1.0000


## 5. Tuning VaDeSC

Until now we always used the paper settings. Maybe they just don't fit our data, so here we try 18 settings: number of clusters (2, 4, 6), size of the latent space (8, 16) and the survival distribution of the clusters (Weibull with k = 2 like the paper, Weibull with a learned k, or log-normal).

On every split we train each setting with 2 seeds and keep the one with the best C-index on the validation set. Then we train only that setting with 5 seeds and test it on the test set.

In [8]:
GRID = [dict(K=K, d_lat=dl, **dist) for K in ([2, 4] if QUICK else [2, 4, 6]) for dl in [8, 16]
        for dist in [dict(weibull_k=2.0), dict(weibull_k=1.0, model_cls=VaDeSCLearnK), dict(dist="lognormal")]]


def name_of(cfg):
    dist = "log-normal" if cfg.get("dist") == "lognormal" else ("Weibull, learned k" if "model_cls" in cfg else "Weibull, k = 2")
    return f"K={cfg['K']}, d={cfg['d_lat']}, {dist}"


tune = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=2)(
    delayed(run_vadesc)(data[(sp, "median/mode")], s, name_of(cfg), sp, **cfg) for sp in SPLITS for s in TUNE_SEEDS for cfg in GRID))
best = tune.groupby(["split", "method"])["val C-index"].mean().groupby("split").idxmax().map(lambda x: x[1])
print("chosen setting per split:")
print(best.value_counts())
cfg_of = {name_of(c): c for c in GRID}
tuned = pd.DataFrame(Parallel(n_jobs=N_JOBS)(
    delayed(run_vadesc)(data[(sp, "median/mode")], s, "VaDeSC, tuned", sp, **cfg_of[best[sp]]) for sp in SPLITS for s in SEEDS))
tuned["setting"] = tuned.split.map(best)
pd.concat([tune.assign(stage="tuning"), tuned.assign(stage="test")]).to_csv("../results/support10_tuning.csv", index=False)
tuned.groupby("method")[["C-index", "IBS", "k"]].agg(["mean", "std"]).round(3)

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed:  2.8min


[Parallel(n_jobs=8)]: Done 146 tasks      | elapsed: 13.6min


[Parallel(n_jobs=8)]: Done 360 out of 360 | elapsed: 32.4min finished


chosen setting per split:
val C-index
K=2, d=16, log-normal            5
K=4, d=16, log-normal            4
K=6, d=16, Weibull, learned k    1
Name: count, dtype: int64


C-index           IBS             k       
                 mean    std   mean    std   mean    std
method                                                  
VaDeSC, tuned     0.7  0.014  0.191  0.005  0.718  0.103

In [9]:
# test C-index of every grid setting (2 seeds), only for reference: the choice above used the validation set
tune.groupby("method")[["val C-index", "C-index", "k"]].mean().sort_values("val C-index", ascending=False).round(3)

,val C-index,C-index,k
method,,,
"K=2, d=16, log-normal",0.697,0.704,NaN
"K=4, d=16, log-normal",0.692,0.695,NaN
"K=4, d=16, Weibull, learned k",0.685,0.689,0.591
"K=2, d=16, Weibull, learned k",0.678,0.681,0.591
"K=2, d=16, Weibull, k = 2",0.674,0.681,2.000
"K=6, d=16, Weibull, learned k",0.672,0.675,0.810
"K=2, d=8, log-normal",0.670,0.669,NaN
"K=6, d=16, log-normal",0.667,0.672,NaN
"K=4, d=8, log-normal",0.662,0.665,NaN


In [10]:
# tuned VaDeSC vs. the paper configuration and vs. the classical baselines, one value per split
allm = pd.concat([by_split(tuned), by_split(miss)[["VaDeSC, median/mode"]], by_split(cls)], axis=1)
print(allm.mean().round(3).to_string())
comps_t = [("VaDeSC, tuned", "VaDeSC, median/mode"), ("Cox PH", "VaDeSC, tuned"), ("RSF", "VaDeSC, tuned"),
           ("DSM", "VaDeSC, tuned"), ("Weibull AFT", "VaDeSC, tuned")]
pd.DataFrame([split_test(allm, a, b, len(comps_t)) for a, b in comps_t]).round(4)

method
VaDeSC, tuned          0.700
VaDeSC, median/mode    0.658
Cox PH                 0.731
DSM                    0.734
RSF                    0.727
Weibull AFT            0.728


,comparison,mean diff,95% CI,higher in,p-value,p (Bonferroni)
0,"VaDeSC, tuned minus VaDeSC, median/mode",0.0420,"[+0.033, +0.051]",10/10 splits,0.0000,0.0000
1,"Cox PH minus VaDeSC, tuned",0.0305,"[+0.021, +0.040]",10/10 splits,0.0000,0.0002
2,"RSF minus VaDeSC, tuned",0.0273,"[+0.018, +0.037]",10/10 splits,0.0001,0.0007
3,"DSM minus VaDeSC, tuned",0.0342,"[+0.025, +0.043]",10/10 splits,0.0000,0.0001
4,"Weibull AFT minus VaDeSC, tuned",0.0283,"[+0.019, +0.037]",10/10 splits,0.0001,0.0003


## Notes

- The classical models are very stable across the 10 splits: C-index 0.727-0.734 (Cox 0.731, DSM 0.734), std about 0.01.
- Without the post-baseline variables every model drops: Cox 0.804 -> 0.731, RSF 0.810 -> 0.727, VaDeSC 0.775 -> 0.663. VaDeSC drops the most. (VaDeSC without them is 0.663 here with 3 seeds and 0.658 in section 4 with 5 seeds.)
- Mean imputation is better than median/mode for VaDeSC (+0.026, better in 10/10 splits). The 2 x 2 test shows that it is the numeric fill that matters (mean instead of median). Giving missing categories their own category makes no real difference. Cox does not care about any of this (0.730-0.732 for every version).
- VAE imputation is better than median/mode (+0.015) but worse than mean imputation (-0.011, Bonferroni p = 0.049, so only just significant).
- The mask did not help. All three mask versions are below mean imputation, which fills in the same values (-0.013 to -0.022, significant after Bonferroni). MissVaDeSC is better than median/mode only because of the mean fill.
- Tuning helps VaDeSC a lot: log-normal or Weibull with learned k and latent size 16 was chosen on every split, and the tuned VaDeSC reaches 0.700 (+0.042 over the paper settings). It is still 0.027-0.034 below every classical model, in 10/10 splits. The learned Weibull k was between 0.54 and 0.81, far from the paper's k = 2.